# WAI Anima / Anima Preview3 for Fooocus on Google Colab

WAI Anima (Anima アーキテクチャの finetune) / Anima Preview3 / Animayume を Google Colab で実行するためのノートブックです。
デフォルトは **WAI Anima v1.0** (https://civitai.com/models/2544636/wai-anima)。設定セルの `MODEL_PROFILE` で他のモデルに切り替え可能です。



In [ ]:
use_cloudflared = False  # gradio share が使えないときだけ True
prepare_runtime = False  # optional for the old bootstrap/diagnostic flow only
run_cli_mode = False  # True runs the headless CLI reproducer instead of Gradio
cli_profile = "official"  # baseline / diagnostic / official / single / worker_pair / worker_single
launch_runtime_flag = "--always-gpu"  # T4/L4 should fit current Anima defaults; switch back to --always-high-vram if VRAM gets tight
preset_variant = "quality_default" #@param ["quality_default", "balanced", "custom_steps"]
custom_steps = 60  #@param {type:"slider", min:1, max:80, step:1}
REPO_URL = "https://github.com/asahiruyoru/Fooocus.git"
REPO_REF = "main"

WAI_ANIMA_BASE_MODEL = {
    "base_preset_name": "wai_anima",
    "default_prompt": "masterpiece, best quality, score_7, highres, safe, 1girl, solo, looking at viewer, smile, long hair, detailed eyes, detailed face, clean lines, smooth shading, soft lighting",
    "default_prompt_negative": "worst quality, low quality, score_1, score_2, score_3, artist name",
    "default_aspect_ratio": "1344*1344",
}
MODEL_PROFILE = "wai_anima"  # wai_anima / anima_preview3 / anima_preview2 / animayume_v03
MODEL_PROFILES = {
    "wai_anima": {
        "display_name": "WAI Anima v1.0",
        "model_page_url": "https://civitai.com/models/2544636/wai-anima",
        "checkpoint_file": "waiANIMA_v10.safetensors",
        "checkpoint_url": "https://civitai.com/api/download/models/2859702?type=Model&format=SafeTensor&size=pruned&fp=fp16",
    },
    "anima_preview3": {
        "display_name": "Anima Preview3",
        "checkpoint_file": "anima-preview3-base.safetensors",
        "checkpoint_url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview3-base.safetensors",
    },
    "anima_preview2": {
        "display_name": "Anima Preview2",
        "checkpoint_file": "anima-preview2.safetensors",
        "checkpoint_url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview2.safetensors",
    },
    "animayume_v03": {
        "display_name": "Animayume v0.3",
        "model_page_url": "https://civitai.com/models/2385278/animayume",
        "checkpoint_file": "AnimaYumev03.safetensors",
        "checkpoint_url": "https://civitai.com/api/download/models/2798563?type=Model&format=SafeTensor&size=full&fp=bf16",
    },
}

MODEL = dict(WAI_ANIMA_BASE_MODEL)
MODEL.update(MODEL_PROFILES[MODEL_PROFILE])
ANIMA_CHECKPOINT_FILE = MODEL["checkpoint_file"]
ANIMA_CHECKPOINT_URL = MODEL["checkpoint_url"]
DEFAULT_ASPECT_RATIO = MODEL["default_aspect_ratio"]
DEFAULT_PROMPT = MODEL["default_prompt"]
DEFAULT_NEGATIVE_PROMPT = MODEL["default_prompt_negative"]
BASE_PRESET_NAME = MODEL["base_preset_name"]
TARGET_PRESET_NAME = MODEL_PROFILE
PRESET_VARIANTS = {
    "quality_default": {"suffix": "", "performance": "Quality", "overwrite_step": -1},
    "balanced": {"suffix": "_balanced", "performance": "Balanced", "overwrite_step": -1},
    "custom_steps": {"suffix": "_custom", "performance": "Quality", "overwrite_step": custom_steps},
}
VARIANT = PRESET_VARIANTS[preset_variant]
LAUNCH_PRESET_NAME = f"{TARGET_PRESET_NAME}{VARIANT['suffix']}"
DEFAULT_PERFORMANCE = VARIANT["performance"]
DEFAULT_OVERWRITE_STEP = VARIANT["overwrite_step"]

print(f"Using model profile: {MODEL_PROFILE} -> {MODEL['display_name']}")
print(f"Preset base: {BASE_PRESET_NAME}")
print(f"Launch preset: {LAUNCH_PRESET_NAME}")
print(f"Preset variant: {preset_variant}")
print(f"Performance: {DEFAULT_PERFORMANCE}")
print(f"Overwrite step: {DEFAULT_OVERWRITE_STEP}")
if "model_page_url" in MODEL:
    print(f"Model page: {MODEL['model_page_url']}")


## 1. GPU 確認

Colab の `Runtime > Change runtime type` で GPU を有効化してから実行してください。


In [ ]:
!nvidia-smi

## 2. Fooocus を取得

fork と branch を設定セルで切り替えられます。


In [ ]:
!pip install -q pygit2==1.15.1 gdown
%cd /content
!rm -rf /content/Fooocus
!git clone --depth 1 --branch {REPO_REF} {REPO_URL} /content/Fooocus
%cd /content/Fooocus


## 2.5. Colab 依存関係を調整

`rembg -> pymatting -> cupy` の import error を避けるため、`numpy` と `cupy` の組み合わせを固定します。


In [ ]:
!pip uninstall -y cupy-cuda12x
!pip install -q --no-cache-dir numpy==1.26.4 "cupy-cuda12x<14"


## 3. モデル配置先を準備

配置先 (Anima 系モデル共通):
- 選択中のチェックポイント (例: `waiANIMA_v10.safetensors`) -> `models/checkpoints/`
- `qwen_3_06b_base.safetensors` (テキストエンコーダ) -> `models/clip/`
- `qwen_image_vae.safetensors` (VAE) -> `models/vae/`

> Anima アーキテクチャ (DiT) のモデルは **必ず** Qwen テキストエンコーダ + Qwen Image VAE をペアでロードします。


In [ ]:
!mkdir -p /content/Fooocus/models/checkpoints
!mkdir -p /content/Fooocus/models/clip
!mkdir -p /content/Fooocus/models/vae


## 4. `wget` でモデル取得

Hugging Face token は使わず、`resolve/main/...` の URL を `wget` で直接取得します。


In [ ]:
%cd /content/Fooocus/models/checkpoints
print(f"Downloading checkpoint: {ANIMA_CHECKPOINT_FILE}")
!wget -nc --content-disposition "{ANIMA_CHECKPOINT_URL}"

%cd /content/Fooocus/models/clip
!wget -nc --content-disposition {QWEN_TEXT_ENCODER_URL}

%cd /content/Fooocus/models/vae
!wget -nc --content-disposition {QWEN_VAE_URL}


## 5. 配置確認


In [ ]:
!ls -lh /content/Fooocus/models/checkpoints
!ls -lh /content/Fooocus/models/clip
!ls -lh /content/Fooocus/models/vae


## 6. `config.txt` を Colab 用に固定


In [ ]:
import json

config = {
    "path_checkpoints": ["/content/Fooocus/models/checkpoints"],
    "path_loras": ["/content/Fooocus/models/loras"],
    "path_embeddings": "/content/Fooocus/models/embeddings",
    "path_vae_approx": "/content/Fooocus/models/vae_approx",
    "path_vae": "/content/Fooocus/models/vae",
    "path_upscale_models": "/content/Fooocus/models/upscale_models",
    "path_inpaint": "/content/Fooocus/models/inpaint",
    "path_controlnet": "/content/Fooocus/models/controlnet",
    "path_clip_vision": "/content/Fooocus/models/clip_vision",
    "path_fooocus_expansion": "/content/Fooocus/models/prompt_expansion/fooocus_expansion",
    "path_wildcards": "/content/Fooocus/wildcards",
    "path_safety_checker": "/content/Fooocus/models/safety_checker",
    "path_sam": "/content/Fooocus/models/sam",
    "path_outputs": "/content/Fooocus/outputs"
}

with open('/content/Fooocus/config.txt', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print(open('/content/Fooocus/config.txt', 'r', encoding='utf-8').read())


In [ ]:
import json
from pathlib import Path

base_json_path = f"/content/Fooocus/presets/{BASE_PRESET_NAME}.json"
quality_json_path = f"/content/Fooocus/presets/{TARGET_PRESET_NAME}.json"
balanced_json_path = f"/content/Fooocus/presets/{TARGET_PRESET_NAME}_balanced.json"
custom_json_path = f"/content/Fooocus/presets/{TARGET_PRESET_NAME}_custom.json"
base_preset_path = Path(base_json_path)
using_repo_preset = base_preset_path.exists()

if using_repo_preset:
    with base_preset_path.open("r", encoding="utf-8") as f:
        base_json_data = json.load(f)
else:
    base_json_data = {
        "default_model": ANIMA_CHECKPOINT_FILE,
        "default_refiner": "None",
        "default_refiner_switch": 0.5,
        "default_vae": "qwen_image_vae.safetensors",
        "default_loras": [[True, "None", 1.0] for _ in range(5)],
        "default_cfg_scale": 4.5,
        "default_sample_sharpness": 2.0,
        "default_sampler": "euler_ancestral",
        "default_scheduler": "simple",
        "default_performance": "Quality",
        "default_advanced_checkbox": True,
        "default_image_number": 32,
        "default_save_metadata_to_images": True,
        "default_metadata_scheme": "a1111",
        "default_prompt": DEFAULT_PROMPT,
        "default_prompt_negative": DEFAULT_NEGATIVE_PROMPT,
        "default_styles": [],
        "default_aspect_ratio": DEFAULT_ASPECT_RATIO,
        "default_overwrite_step": -1,
        "default_overwrite_switch": -1,
        "default_inpaint_engine_version": "None",
        "checkpoint_downloads": {ANIMA_CHECKPOINT_FILE: ANIMA_CHECKPOINT_URL},
        "embeddings_downloads": {},
        "lora_downloads": {},
        "vae_downloads": {},
        "previous_default_models": [ANIMA_CHECKPOINT_FILE]
    }

base_json_data = dict(base_json_data)
base_json_data["default_model"] = ANIMA_CHECKPOINT_FILE
base_json_data["default_prompt"] = DEFAULT_PROMPT
base_json_data["default_prompt_negative"] = DEFAULT_NEGATIVE_PROMPT
base_json_data["default_aspect_ratio"] = DEFAULT_ASPECT_RATIO
base_json_data["checkpoint_downloads"] = {ANIMA_CHECKPOINT_FILE: ANIMA_CHECKPOINT_URL}
base_json_data["previous_default_models"] = [ANIMA_CHECKPOINT_FILE]

balanced_json_data = dict(base_json_data)
balanced_json_data["default_performance"] = "Balanced"

custom_json_data = dict(base_json_data)
custom_json_data["default_performance"] = "Quality"
custom_json_data["default_overwrite_step"] = custom_steps

launch_json_path = {
    "quality_default": quality_json_path,
    "balanced": balanced_json_path,
    "custom_steps": custom_json_path,
}[preset_variant]
launch_json_data = {
    "quality_default": base_json_data,
    "balanced": balanced_json_data,
    "custom_steps": custom_json_data,
}[preset_variant]

if preset_variant == "quality_default" and using_repo_preset and quality_json_path == base_json_path:
    print(f"Using tracked preset: {base_json_path}")
else:
    with open(launch_json_path, "w", encoding="utf-8") as f:
        json.dump(launch_json_data, f, ensure_ascii=False, indent=4, sort_keys=True)
    print(f"Wrote launch preset: {launch_json_path}")

print(json.dumps(launch_json_data, ensure_ascii=False, indent=4, sort_keys=True))


## 7. Fooocus を起動

 と  の両方に対応しています。


In [ ]:
import re
import subprocess
import threading
import time

%cd /content/Fooocus

if prepare_runtime:
    !python scripts/anima_preview2_colab_bootstrap.py --prepare-only

if run_cli_mode:
    !python scripts/anima_preview2_colab_bootstrap.py --profile {cli_profile}
elif not use_cloudflared:
    !python entry_with_update.py --preset {LAUNCH_PRESET_NAME} --share {launch_runtime_flag}
else:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

    proc = subprocess.Popen(
        ["python", "entry_with_update.py", "--preset", LAUNCH_PRESET_NAME, launch_runtime_flag],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    port_holder = {"port": None}

    def reader():
        for line in proc.stdout:
            print(line, end="")
            match = re.search(r"Running on local URL:\s+http://127\.0\.0\.1:(\d+)", line)
            if match and port_holder["port"] is None:
                port_holder["port"] = match.group(1)

    t = threading.Thread(target=reader, daemon=True)
    t.start()

    for _ in range(180):
        if port_holder["port"] is not None:
            break
        if proc.poll() is not None:
            raise RuntimeError("Fooocus process ended before cloudflared tunnel was created")
        time.sleep(1)

    if port_holder["port"] is None:
        raise RuntimeError("Could not detect local Fooocus port")

    print(f"Launching cloudflared for port {port_holder['port']}")
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port_holder['port']}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    public_url = None
    for _ in range(120):
        line = tunnel.stdout.readline()
        if not line:
            time.sleep(1)
            continue
        print(line, end="")
        match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

    if public_url is None:
        raise RuntimeError("Could not detect cloudflared public URL")

    print(f"Public URL: {public_url}")
    proc.wait()
